# WEEK 6 LAB ADDITIONAL NOTEBOOK — Predictive Analytics I

### With statsmodels OLS + Logit + VIF + Cook's Distance

In [ ]:
!pip install scikit-learn==1.5.2 # installing sklearn

In [ ]:
!pip install statsmodels==0.14.4

In [ ]:
# import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, confusion_matrix, classification_report, roc_curve, auc
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import scipy.stats as stats

np.random.seed(42) # setting for replication

### 1. Create dataset

In [ ]:
n = 500
Income = np.random.normal(50000, 12000, n)
DebtRatio = np.random.uniform(0.1, 0.6, n)
Age = np.random.normal(40, 10, n)

CreditScore = (
    700 
    + 0.0008 * Income 
    - 120 * DebtRatio 
    + 0.5 * Age 
    + np.random.normal(0, 25, n)
)

Default = (CreditScore < 680).astype(int)

df = pd.DataFrame({
    "Income": Income,
    "DebtRatio": DebtRatio,
    "Age": Age,
    "CreditScore": CreditScore,
    "Default": Default
})

### 2. OLS Regression

In [ ]:
ols_model = smf.ols("CreditScore ~ Income + DebtRatio + Age", data=df).fit()
print(ols_model.summary())

rmse = np.sqrt(mean_squared_error(df["CreditScore"], ols_model.fittedvalues))
print("\nRMSE:", rmse)


### 3. Diagnostics

In [ ]:
residuals = ols_model.resid

plt.figure(figsize=(6,4))
sns.scatterplot(x=ols_model.fittedvalues, y=residuals)
plt.axhline(0, color='red')
plt.title("Residual Plot")
plt.show()

sm.qqplot(residuals, line='45', fit=True)
plt.title("QQ Plot of Residuals")
plt.show()

In [ ]:
# VIF
X = df[["Income", "DebtRatio", "Age"]]
X_with_const = sm.add_constant(X)

vif_df = pd.DataFrame()
vif_df["Variable"] = X_with_const.columns
vif_df["VIF"] = [variance_inflation_factor(X_with_const.values, i) 
                 for i in range(X_with_const.shape[1])]
print("\nVIF Table:")
print(vif_df)

In [ ]:
# Cook's Distance
influence = ols_model.get_influence()
cooks_d = influence.cooks_distance[0]

plt.figure(figsize=(6,4))
plt.stem(cooks_d, markerfmt=",")
plt.title("Cook's Distance")
plt.xlabel("Observation")
plt.ylabel("Cook's Distance")
plt.show()

### 4. Logistic Regression

In [ ]:
logit_model = smf.logit("Default ~ Income + DebtRatio + Age", data=df).fit()
print(logit_model.summary())



In [ ]:
# Odds ratios
odds_ratios = pd.DataFrame({
    "Odds Ratio": logit_model.params.apply(np.exp),
    "p-value": logit_model.pvalues
})
print("\nOdds Ratios:")
print(odds_ratios)

In [ ]:
# Marginal effects
mfx = logit_model.get_margeff()
print("\nMarginal Effects:")
print(mfx.summary())

### 5. Classification Metrics

In [ ]:
df["pred_prob"] = logit_model.predict(df)
df["pred_class"] = (df["pred_prob"] > 0.5).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(df["Default"], df["pred_class"]))

print("\nClassification Report:")
print(classification_report(df["Default"], df["pred_class"]))

### 6. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(df["Default"], df["pred_prob"])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()



### 7. Further things to think about

In [ ]:

print("""
1. Interpret the VIF table. Which variables show multicollinearity?
2. Identify observations with high Cook’s distance. Should they be removed?
3. Compare RMSE and MAE. Which is more robust?
4. Interpret the odds ratios from the logit model.
5. Interpret the marginal effects. Which variable has the strongest effect?
6. Recreate the ROC curve using only two predictors.
7. Try a different threshold (0.3, 0.7) and compare confusion matrices.
""")